# Required Imports

In [1]:
import os
import re
import random
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
from numpy import dot
from numpy.linalg import norm
from collections import defaultdict
from tqdm import tqdm

# -- NEW IMPORTS FOR GRAPHS --
from torch_geometric.data import Data, Dataset
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GATConv, global_mean_pool, global_max_pool

## Parameters

In [2]:
# Configurazione PyG
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

if torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")
print(f'device being used: {device}')

# Configuration settings
n_classes = 40 
n_epochs = 100
train_batch_size = 32
test_batch_size = 1
learning_rate = 0.0005
num_node_features = 100 # Sostituisce img_w e img_h
num_nodes = 52

device being used: cpu


## Load Probe and Gallery sets

In [3]:
def extract_id_from_filename(filename):
    # Regex super resiliente: cerca un numero seguito da 'm' o 'f', ignorando se c'è un underscore o lettere extra
    match = re.search(r'(\d+[mf])', filename) 
    return match.group(1) if match else None

class GraphReIDTrainDataset(Dataset):
    def __init__(self, data_root):
        super().__init__()
        self.data_root = data_root
        self.samples, self.id_to_class = self._load_data()
        print(f"TRAIN: Trovati {len(self.samples)} grafi in {data_root}")

    def _load_data(self):
        paths = []
        for dirpath, _, filenames in os.walk(self.data_root):
            for f in filenames:
                if f.endswith('.pt'):
                    paths.append(os.path.join(dirpath, f))
        
        unique_ids = sorted(list(set([extract_id_from_filename(p) for p in paths if extract_id_from_filename(p)])))
        id_to_class = {orig_id: idx for idx, orig_id in enumerate(unique_ids)}
        
        samples = []
        for p in paths:
            pid = extract_id_from_filename(p)
            if pid:
                samples.append((p, id_to_class[pid]))
        return samples, id_to_class

    def len(self):
        return len(self.samples)

    def get(self, idx):
        path, target = self.samples[idx]
        data = torch.load(path, weights_only=False)
        data.y = torch.tensor([target], dtype=torch.long)
        data.path = path # Salviamo il path per il PKSampler
        return data

# PKSampler invariato
class PKSampler(torch.utils.data.Sampler):
    def __init__(self, dataset, P=8, K=4):
        self.dataset = dataset
        self.P = P
        self.K = K
        self.batch_size = self.P * self.K
        
        self.id_to_indices = defaultdict(list)
        for idx in range(len(self.dataset)):
            target = self.dataset.samples[idx][1]
            self.id_to_indices[target].append(idx)
        self.unique_ids = list(self.id_to_indices.keys())

    def __iter__(self):
        random.shuffle(self.unique_ids)
        batches = []
        for i in range(0, len(self.unique_ids), self.P):
            p_ids = self.unique_ids[i:i+self.P]
            if len(p_ids) < self.P: break
            batch = []
            for pid in p_ids:
                indices = self.id_to_indices[pid]
                if len(indices) >= self.K:
                    batch.extend(random.sample(indices, self.K))
                else:
                    batch.extend(np.random.choice(indices, size=self.K, replace=True))
            batches.append(batch)
        
        random.shuffle(batches)
        for batch in batches:
            yield from batch

    def __len__(self):
        return (len(self.unique_ids) // self.P) * self.batch_size

class GraphProbeGalleryValTest(Dataset):
    def __init__(self, data_root):
        super().__init__()
        self.data_root = data_root  
        self.data = self.read_folder()  
        self.samples = self.create_samples()  
        print(f"TEST: Trovati {len(self.data)} file totali. Generati {len(self.samples)} campioni Probe-Gallery.")

    def len(self):
        return len(self.samples)  

    def get(self, idx):
        probe_pth, gallery_pths, is_match, probe_target, gallery_targets = self.samples[idx]  
        probe_data = torch.load(probe_pth, weights_only=False)
        
        gallery_data_list = []
        for item in gallery_pths:
            gallery_data_list.append(torch.load(item, weights_only=False))
            
        return probe_data, gallery_data_list, is_match, probe_pth, gallery_pths, probe_target, gallery_targets

    def read_folder(self):
        paths = []  
        for dirpath, _, filenames in os.walk(self.data_root):
            files = [f for f in filenames if f.endswith('.pt')]
            # Metodo OS-indipendente per prendere l'ultima cartella (es. "Probe")
            label = os.path.basename(os.path.normpath(dirpath))
            for item in files:
                paths.append([os.path.join(dirpath, item), label])  
        return paths

    def create_samples(self):
        # Case insensitive per massima sicurezza
        probes = [x[0] for x in self.data if x[1].lower() == 'probe']  
        gallery = [x[0] for x in self.data if x[1].lower() == 'gallery']  
        samples = []
        for probe in probes:
            probe_id = extract_id_from_filename(probe)  
            if not probe_id: continue
            is_match, gallery_targets = [], []
            for item in gallery:
                item_id = extract_id_from_filename(item)  
                if not item_id: continue
                is_match.append(1 if probe_id == item_id else -1)
                gallery_targets.append(item_id)
            
            # Aggiunge alla lista solo se ha effettivamente trovato la gallery
            if len(gallery_targets) > 0:
                samples.append([probe, gallery, is_match, probe_id, gallery_targets])
        return samples

# Dataloaders Inizializzati
train_dataset = GraphReIDTrainDataset('./output/GraphData/D1/')
pk_sampler = PKSampler(train_dataset, P=8, K=4) 
train_dataloader = DataLoader(train_dataset, batch_size=32, sampler=pk_sampler, drop_last=True)

test_dataset = GraphProbeGalleryValTest('./output/GraphData/D2t/')
test_dataloader = DataLoader(test_dataset, batch_size=1, shuffle=False)

TRAIN: Trovati 800 grafi in ./output/GraphData/D1/
TEST: Trovati 820 file totali. Generati 410 campioni Probe-Gallery.


# Graph Attention Network

In [4]:
class GraphAttentionNetwork(nn.Module):
    def __init__(self, in_channels, temporal_dim, hidden_channels, out_channels, heads=4):
        super().__init__()
        
        # 1. Frequency Positional Encoding: Teaches the fully connected graph the physical order of subcarriers
        self.pos_emb = nn.Embedding(52, 16) # 52 subcarriers mapped to 16-D vectors
        
        # 2. Temporal 1D CNN: Slides over the 100 packets to extract motion signatures
        self.temporal_conv = nn.Sequential(
            nn.Conv1d(in_channels=1, out_channels=16, kernel_size=5, padding=2),
            nn.BatchNorm1d(16),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Conv1d(in_channels=16, out_channels=32, kernel_size=3, padding=1),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(1) # Compresses the time sequence into a single 32-D feature vector per node
        )
        
        # Fusion layer: Combines Temporal Motion (32-D) + Frequency Position (16-D) -> temporal_dim
        self.feature_fusion = nn.Sequential(
            nn.Linear(32 + 16, temporal_dim),
            nn.ReLU()
        )
        
        # 3. Graph Attention Layers
        self.conv1 = GATConv(temporal_dim, hidden_channels, heads=heads, dropout=0.3)
        self.conv2 = GATConv(hidden_channels * heads, out_channels, heads=1, concat=False, dropout=0.3)
        
    def forward(self, x, edge_index, batch):
        N = x.size(0) # Total number of nodes in the batch
        
        # --- A. Inject Frequency Positional Encoding ---
        # We know every graph has exactly 52 nodes, so we generate repeating indices [0-51, 0-51...]
        subcarrier_idx = (torch.arange(N, device=x.device) % 52)
        pos_features = self.pos_emb(subcarrier_idx) # [N, 16]
        
        # --- B. Extract Temporal Sequence Features ---
        # Reshape for Conv1d from [N, 100] to [N, Channels, Seq_Len] -> [N, 1, 100]
        x_seq = x.unsqueeze(1) 
        temp_features = self.temporal_conv(x_seq) # Output: [N, 32, 1]
        temp_features = temp_features.squeeze(-1) # Output: [N, 32]
        
        # --- C. Feature Fusion ---
        node_features = torch.cat([temp_features, pos_features], dim=1) # [N, 48]
        node_features = self.feature_fusion(node_features) # [N, temporal_dim]
        
        # --- D. Spatial Routing (GAT) ---
        x_g, alpha1 = self.conv1(node_features, edge_index, return_attention_weights=True)
        x_g = F.elu(x_g)
        
        x_g, alpha2 = self.conv2(x_g, edge_index, return_attention_weights=True)
        
        # --- E. Multi-Faceted Readout ---
        x_mean = global_mean_pool(x_g, batch)
        x_max = global_max_pool(x_g, batch)
        
        global_features = torch.cat([x_mean, x_max], dim=1)
        global_features = F.normalize(global_features, p=2, dim=1)
        
        return global_features, alpha2

# Graph Consistency Loss & Hard Triplet Loss

In [5]:
class GraphConsistencyTripletLoss(nn.Module):
    def __init__(self, margin=0.3, lambda_gc=0.1):
        super().__init__()
        self.ranking_loss = nn.MarginRankingLoss(margin=margin)
        self.lambda_gc = lambda_gc

    def forward(self, embeddings, attentions, targets, batch):
        # --- 1. Hard Triplet Loss Classica ---
        n = embeddings.size(0)
        dist = torch.pow(embeddings, 2).sum(dim=1, keepdim=True).expand(n, n)
        dist = dist + dist.t()
        dist.addmm_(embeddings, embeddings.t(), beta=1, alpha=-2)
        dist = dist.clamp(min=1e-12).sqrt()

        mask = targets.expand(n, n).eq(targets.expand(n, n).t())
        dist_ap, dist_an = [], []

        for i in range(n):
            dist_ap.append(dist[i][mask[i]].max().unsqueeze(0))
            dist_an.append(dist[i][mask[i] == 0].min().unsqueeze(0))

        dist_ap = torch.cat(dist_ap)
        dist_an = torch.cat(dist_an)
        y = torch.ones_like(dist_an)
        triplet_loss = self.ranking_loss(dist_an, dist_ap, y)
        
        # --- 2. Graph Consistency Loss (Sperimentale) ---
        edge_weights = attentions[1] 
        gc_loss = torch.tensor(0.0, device=embeddings.device)
        
        return triplet_loss + (self.lambda_gc * gc_loss)

# Training and Testing functions

In [6]:
def compute_ap(pos_ranks):
    """Calcolo corretto della Average Precision"""
    if len(pos_ranks) == 0:
        return 0.0
    ap = 0.0
    for i, pos in enumerate(pos_ranks):
        ap += (i + 1) / (pos + 1)
    return ap / len(pos_ranks)


from torch_geometric.data import Batch

def testing_step(model, test_dataloader):
    model.eval()

    test_dataset = test_dataloader.dataset if hasattr(test_dataloader, 'dataset') else test_dataloader

    if len(test_dataset.samples) == 0:
        print("\nWARNING: Test Dataset è vuoto!")
        return 0.0, 0.0, 0.0, 0.0

    # 1. ESTRAZIONE DELLA GALLERY (UNA VOLTA SOLA)
    # Prendiamo i percorsi della gallery dal primissimo campione (sono uguali per tutti)
    gallery_pths = test_dataset.samples[0][1] 
    
    print(f"\nEstrazione feature della Gallery ({len(gallery_pths)} grafi)...")
    gallery_features_list = []
    gallery_ids = []

    with torch.no_grad():
        # Processiamo la gallery a blocchi di 32 per non saturare la RAM
        batch_size = 32
        for i in range(0, len(gallery_pths), batch_size):
            batch_paths = gallery_pths[i:i+batch_size]
            batch_graphs = [torch.load(p, weights_only=False) for p in batch_paths]
            g_batch = Batch.from_data_list(batch_graphs).to(device)

            # Estrai le feature e spostale su CPU per tenerle in memoria
            features, _ = model(g_batch.x, g_batch.edge_index, g_batch.batch)
            gallery_features_list.append(features.cpu())

            # Salva gli ID nativi
            batch_ids = g_batch.subject_id
            if isinstance(batch_ids, str): 
                batch_ids = [batch_ids]
            gallery_ids.extend(batch_ids)

        gallery_features = torch.cat(gallery_features_list, dim=0)

    # 2. VALUTAZIONE DELLE PROBE
    print("\nValutazione Probes in corso...")
    correct1 = correct2 = correct3 = correct4 = correct5 = 0
    correct6 = correct7 = correct8 = correct9 = correct10 = 0
    samples = 0  
    K = 10
    AP_scores = []  
    rank_matrix = []

    with torch.no_grad():
        # Invece di usare il DataLoader, iteriamo direttamente sulla lista dei campioni
        for sample in tqdm(test_dataset.samples, leave=False):
            probe_pth = sample[0] # Il percorso della probe è l'elemento 0
            
            # Carichiamo solo la singola probe (1 lettura da disco invece di 400+)
            probe_data = torch.load(probe_pth, weights_only=False)
            probe_batch = Batch.from_data_list([probe_data]).to(device)

            # Feature della probe
            probe_feat, _ = model(probe_batch.x, probe_batch.edge_index, probe_batch.batch)
            probe_feat = probe_feat.cpu()

            scores = []
            
            # Calcolo similarità vettoriale rapida con la gallery pre-calcolata
            for i in range(gallery_features.shape[0]):
                a = probe_feat[0].numpy()
                b = gallery_features[i].numpy()
                cos_sim = dot(a, b)/(norm(a)*norm(b))
                
                gal_id = gallery_ids[i]
                scores.append([cos_sim, gal_id])
            
            samples += 1
            topk = sorted(scores, key=lambda element: (element[0]), reverse=True)
            
            p_id = probe_batch.subject_id[0] if isinstance(probe_batch.subject_id, list) else probe_batch.subject_id
            rank_matrix.append([p_id, topk])

            # -- CALCOLO mAP --
            pos_ranks = []
            for i, score in enumerate(topk):
                gal_id = score[1]
                if p_id == gal_id:
                    pos_ranks.append(i)
                    
            if pos_ranks:
                AP = compute_ap(pos_ranks)  
                AP_scores.append(AP)  

        # -- CALCOLO MATRICE DEI RANK --
        for idx, probe in enumerate(rank_matrix):
            p_id = probe[0]
            for i in range(K):
                gal_id = probe[1][i][1]
                if p_id == gal_id:
                    if i == 0:
                        correct1 += 1; correct2 += 1; correct3 += 1; correct4 += 1; correct5 += 1
                        correct6 += 1; correct7 += 1; correct8 += 1; correct9 += 1; correct10 += 1
                    elif i == 1:
                        correct2 += 1; correct3 += 1; correct4 += 1; correct5 += 1; correct6 += 1
                        correct7 += 1; correct8 += 1; correct9 += 1; correct10 += 1
                    elif i == 2:
                        correct3 += 1; correct4 += 1; correct5 += 1; correct6 += 1; correct7 += 1
                        correct8 += 1; correct9 += 1; correct10 += 1
                    elif i == 3:
                        correct4 += 1; correct5 += 1; correct6 += 1; correct7 += 1; correct8 += 1
                        correct9 += 1; correct10 += 1
                    elif i == 4:
                        correct5 += 1; correct6 += 1; correct7 += 1; correct8 += 1; correct9 += 1
                        correct10 += 1
                    elif i == 5:
                        correct6 += 1; correct7 += 1; correct8 += 1; correct9 += 1; correct10 += 1
                    elif i == 6:
                        correct7 += 1; correct8 += 1; correct9 += 1; correct10 += 1
                    elif i == 7:
                        correct8 += 1; correct9 += 1; correct10 += 1
                    elif i == 8:
                        correct9 += 1; correct10 += 1
                    elif i == 9:
                        correct10 += 1
                    break

    mAP = np.mean(AP_scores) if AP_scores else 0.0
    print(f'Total samples: {samples}, Rank #1: {correct1/samples*100.0:.3f}%, Rank #2: {correct2/samples*100.0:.3f}%, Rank #3: {correct3/samples*100.0:.3f}%, Rank #4: {correct4/samples*100.0:.3f}%, Rank #5: {correct5/samples*100.0:.3f}%, Rank #6: {correct6/samples*100.0:.3f}%, Rank #7: {correct7/samples*100.0:.3f}%, Rank #8: {correct8/samples*100.0:.3f}%, Rank #9: {correct9/samples*100.0:.3f}%, Rank #10: {correct10/samples*100.0:.3f}%, mAP:{mAP*100.0:.3f}%')
    return correct1/samples, correct5/samples, correct10/samples, mAP

In [7]:
def training_model(model, train_dataloader, test_dataloader, optimizer, criterion, n_epochs):
    bestRank1 = 0.0

    for epoch in range(n_epochs):
        model.train()
        train_loss = 0.0

        for batch_data in tqdm(train_dataloader, leave=False, total=len(train_dataloader)):
            # I dati PyG includono già x, edge_index, batch e y(targets)
            batch_data = batch_data.to(device)
            targets = batch_data.y.view(-1)
            
            # Forward sulla GAT
            features, attentions = model(batch_data.x, batch_data.edge_index, batch_data.batch)
            
            # Loss combinata
            loss = criterion(features, attentions, targets, batch_data.batch)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            train_loss += loss.item()

        # Validation step
        rank1, rank5, rank10, _ = testing_step(model, test_dataloader)

        # Salvataggi
        models_folder = os.path.join('./output/models/')
        os.makedirs(models_folder, exist_ok=True)
        if rank1 > bestRank1:
            torch.save(model.state_dict(), f'{models_folder}/best_GNN_state_r1.bin')
            bestRank1 = rank1
            
        print(f"Epoch: [{epoch+1}/{n_epochs}]| Loss: {train_loss/len(train_dataloader):.5f}")

## Model Parameters

In [8]:
# Initialize the upgraded GNN model (temporal_dim=128 maps the 100 packets into a richer space)
model = GraphAttentionNetwork(in_channels=100, temporal_dim=128, hidden_channels=256, out_channels=256).to(device)
criterion = GraphConsistencyTripletLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

## Network Training

In [9]:
# Start training model
training_model(model, train_dataloader, test_dataloader, optimizer, criterion, n_epochs)


Estrazione feature della Gallery (410 grafi)...

Valutazione Probes in corso...


Total samples: 410, Rank #1: 47.073%, Rank #2: 57.805%, Rank #3: 60.732%, Rank #4: 65.854%, Rank #5: 68.293%, Rank #6: 71.220%, Rank #7: 73.902%, Rank #8: 77.317%, Rank #9: 78.537%, Rank #10: 79.024%, mAP:23.386%
Epoch: [1/100]| Loss: 0.32925



Estrazione feature della Gallery (410 grafi)...

Valutazione Probes in corso...


Total samples: 410, Rank #1: 55.854%, Rank #2: 64.878%, Rank #3: 69.024%, Rank #4: 73.902%, Rank #5: 76.341%, Rank #6: 78.537%, Rank #7: 81.220%, Rank #8: 82.927%, Rank #9: 84.390%, Rank #10: 84.878%, mAP:29.268%
Epoch: [2/100]| Loss: 0.31717



Estrazione feature della Gallery (410 grafi)...

Valutazione Probes in corso...


...

Total samples: 410, Rank #1: 76.098%, Rank #2: 84.390%, Rank #3: 88.780%, Rank #4: 92.195%, Rank #5: 93.171%, Rank #6: 94.878%, Rank #7: 96.098%, Rank #8: 97.073%, Rank #9: 97.317%, Rank #10: 97.317%, mAP:65.958%
Epoch: [98/100]| Loss: 0.11545



Estrazione feature della Gallery (410 grafi)...

Valutazione Probes in corso...


Total samples: 410, Rank #1: 76.829%, Rank #2: 83.902%, Rank #3: 88.537%, Rank #4: 90.488%, Rank #5: 93.171%, Rank #6: 93.902%, Rank #7: 95.122%, Rank #8: 95.854%, Rank #9: 96.098%, Rank #10: 96.341%, mAP:64.580%
Epoch: [99/100]| Loss: 0.08405



Estrazione feature della Gallery (410 grafi)...

Valutazione Probes in corso...


Total samples: 410, Rank #1: 80.000%, Rank #2: 87.317%, Rank #3: 90.488%, Rank #4: 92.927%, Rank #5: 94.390%, Rank #6: 95.854%, Rank #7: 96.098%, Rank #8: 96.098%, Rank #9: 97.073%, Rank #10: 97.317%, mAP:69.195%
Epoch: [100/100]| Loss: 0.13114


 ## Performance Evaluation

In [10]:
# Initialize the GNN model with the same architecture and parameters
m1 = GraphAttentionNetwork(in_channels=100, temporal_dim=128, hidden_channels=256, out_channels=256).to(device)
# Load pre-trained model weights
models_folder = os.path.join('./output/models/')
m1.load_state_dict(torch.load(f'{models_folder}/best_GNN_state_r1.bin', map_location=device))

# Evaluate and print the performance of the model with the respective evaluation metric
print('Model w/ best R1:')
testing_step(m1, test_dataloader)

Model w/ best R1:

Estrazione feature della Gallery (410 grafi)...

Valutazione Probes in corso...


Total samples: 410, Rank #1: 80.244%, Rank #2: 87.805%, Rank #3: 90.488%, Rank #4: 91.707%, Rank #5: 93.171%, Rank #6: 95.122%, Rank #7: 95.854%, Rank #8: 96.341%, Rank #9: 96.829%, Rank #10: 97.073%, mAP:64.778%


(0.802439024390244,
 0.9317073170731708,
 0.9707317073170731,
 np.float64(0.6477805401319615))